# Exercise 6: Community exploration

*The Turkic World · version 0.1.0*

> Visit the SIGTURK website, the Apertium Turkic page, or the Universal Dependencies project. Find a recent shared task, release, or publication related to Turkic NLP. Summarise the contribution in 2–3 sentences and explain how it addresses one of the gaps identified in the resource section.

The code lists recently updated Turkic repositories in the Universal Dependencies and Turkic UD GitHub organisations, as a starting point. Always open the release notes: a repository update is not a release.

**Setup.** `pip install "turkicnlp==0.5.0" requests pandas`. Cells that query websites need internet access; if a request fails, the cell records the failure instead of a number. An unavailable value is not zero.

In [ ]:
import datetime
import requests
import pandas as pd

ACCESSED = datetime.date.today().isoformat()        # record the access date with every observation
HEADERS = {"User-Agent": "turkic-nlp-book-exercises/0.1 (educational use)"}

def get_json(url, **params):
    """Return parsed JSON, or {'_error': ...} if the request fails (never a fake value)."""
    try:
        r = requests.get(url, params=params or None, headers=HEADERS, timeout=30)
        r.raise_for_status()
        return r.json()
    except Exception as exc:
        return {"_error": f"{type(exc).__name__}: {exc}"}

def get_text(url):
    try:
        r = requests.get(url, headers=HEADERS, timeout=30)
        r.raise_for_status()
        return r.text
    except Exception as exc:
        return None

def github_org_repos(org):
    """All public repositories of a GitHub organisation (paged listing; avoids the search API's low rate limit)."""
    repos, page = [], 1
    while True:
        d = get_json(f"https://api.github.com/orgs/{org}/repos", per_page=100, page=page)
        if not isinstance(d, list):
            return repos, d.get("_error")
        repos += d
        if len(d) < 100:
            return repos, None
        page += 1

In [ ]:
TURKIC = ["Turkish", "Kazakh", "Kyrgyz", "Uzbek", "Uyghur", "Tatar", "Turkmen", "Azerbaijani",
          "Bashkir", "Sakha", "Yakut", "Chuvash", "Karakalpak", "Kumyk", "Ottoman", "Old_Turkish"]
rows = []
ud, err1 = github_org_repos("UniversalDependencies")
for r in ud:
    if any(r["name"].startswith(f"UD_{name}") for name in TURKIC):
        rows.append({"repository": r["name"], "last push": r["pushed_at"][:10], "url": r["html_url"]})
turkic_ud, err2 = github_org_repos("ud-turkic")
for r in turkic_ud:
    rows.append({"repository": "ud-turkic/" + r["name"], "last push": r["pushed_at"][:10], "url": r["html_url"]})
if err1 or err2:
    print("Some requests failed:", err1 or "", err2 or "")

recent = pd.DataFrame(rows).drop_duplicates("repository")
if recent.empty:
    print("No results (the GitHub API may be rate-limited; try again later or browse the sites directly).")
else:
    display(recent.sort_values("last push", ascending=False).head(15).reset_index(drop=True))

In [ ]:
# Read the changelog of one treebank to find what was actually released and when
REPO = "UD_Turkmen-TUD"
readme = get_text(f"https://raw.githubusercontent.com/UniversalDependencies/{REPO}/master/README.md")
if readme:
    start = readme.find("# Changelog")
    print(readme[start:start + 600] if start >= 0 else readme[:600])
else:
    print("README not retrieved")

## Worked solution (checked 18 October 2026)

**Contribution: the UD-Turkic parallel treebanks** (Eslami et al., 2025; `github.com/ud-turkic`). The project annotates the same 148 grammar-example sentences in Azerbaijani, Kyrgyz, Turkish and Uzbek following Universal Dependencies, released as the TueCL treebanks, with more languages in progress. Because the content is identical across languages, differences between the treebanks show annotation choices rather than different sentences.

**Gap addressed.** It gives several smaller languages their first comparable gold test data and makes inconsistent annotation conventions across Turkic treebanks visible, which is a precondition for fair cross-lingual parsing evaluation (The Turkic World, resource landscape; Neural Morphology).

**Limitation.** The sentences are short grammar examples translated from Turkish, so they say little about performance on natural text. A second example from the listing above: `UD_Turkmen-TUD` records *Initial release in Universal Dependencies* in v2.18 (15 May 2026) — a release, unlike a repository that has only been updated.

## Your turn — extension

Pick an item from the listing (or from SIGTURK or Apertium), open its release notes, and write your 2–3 sentence summary, the date, and the gap it addresses here.